# Processes

**Process**: An instance of a program (e.g a Python interpreter)

```
+ Takes advantage of multiple CPUs and cores
+ Separate memory space -> Memory is not shared between processes
+ Great for CPU-bound processing
+ New process is started independently from other processes
+ Processes are interruptable/killable
+ One GIL for each process -> avoids GIL limitation

- Heavyweight
- Starting a process is slower than starting a thread
- More memory
- IPC (inter-process communication) is more complicated
```

In [ ]:
from multiprocessing import Process
import os
import time

def square_numbers():
  for i in range(100):
    i*i
    time.sleep(0.1)

if __name__ == "__main__":
  processes = []
  num_processes = min(4,os.cpu_count()//2)

  for _ in range(num_processes):
    p = Process(target=square_numbers)
    processes.append(p)

  for p in processes:
    p.start()

  for p in processes:
    p.join()

  print(f"Finish multiprocessing.")


# Multi-processing with shared object and locking

Unlink threads, processes does not live in same memory space. So global variable based sharing is NOT possible.

We use special shared memory objects `Value` & `Array` for sharing data among processes.

## Using `Value` object

In [ ]:
from multiprocessing import Process, Value, Array, Lock
import os
import time

def increase(shared_number, lock):
  time.sleep(0.1)
  with lock:
    shared_number.value += 1

def add_100(shared_number, lock):
  for i in range(100):
    time.sleep(0.1)
    with lock:
      shared_number.value += 1

if __name__=='__main__':
  processes = []
  processes_count = min(3, os.cpu_count()//2)
  lock = Lock()

  shared_number = Value('i', 0)
  print(f"Number at beginning is {shared_number.value}")

  for _ in range(processes_count):
    # p = Process(target=increase, args=(shared_number,lock))
    p = Process(target=add_100, args=(shared_number,lock))
    processes.append(p)

  for p in processes:
    p.start()

  for p in processes:
    p.join()

  print(f"Number at end is {shared_number.value}")

  # when calling add_100(), end value should be equal to "processes_count * 100"
  # when calling increase(), end value should be equal to "processes_count"
  # without lock, race condition can happen.

  print('end main')


## Using `Array` object

In [ ]:
from multiprocessing import Process, Value, Array, Lock
import os
import time

def add_100(shared_array, lock):
  for _ in range(100):
    time.sleep(0.1)
    for i in range(len(shared_array)):
      with lock:
        shared_array[i] += 1

if __name__=='__main__':
  processes = []
  processes_count = min(3, os.cpu_count()//2)
  lock = Lock()

  shared_array = Array('d', [0.0, 100.0, 200.0])
  print(f"Array at beginning is {shared_array[:]}")

  for _ in range(processes_count):
    p = Process(target=add_100, args=(shared_array,lock))
    processes.append(p)

  for p in processes:
    p.start()

  for p in processes:
    p.join()

  print(f"Array at end is {shared_array[:]}")

  # when calling add_100(), end value should be equal to "processes_count * 100"
  # without lock, race condition can happen.

  print('end main')


# Using Queue with threading

Queue are excelling for thread safe & process safe data exchanges in a multi-processing & multi-threading environment.

It is a linear data structure that follows **First-In-First-Out** principle.

In [ ]:
from multiprocessing import Process, Array, Queue, Lock
import time

def square(numbers, q):
  for i in numbers:
    q.put(i**2)

def make_negative(numbers, q):
  for i in numbers:
    q.put(i*-1)

if __name__=='__main__':
  q = Queue()
  numbers = range(1, 6)

  p1 = Process(target=square, args=(numbers, q))
  p2 = Process(target=make_negative, args=(numbers, q))

  p1.start()
  p2.start()

  p1.join()
  p2.join()

  while not q.empty():
    print(q.get())

  print('end main')


# Using Process Pool

In [ ]:
from multiprocessing import Pool

def cube(number):
  return number**3

if __name__=='__main__':
  numbers = range(10)
  pool = Pool()

  # map, apply, join, close
  result = pool.map(cube, numbers)

  pool.close()
  pool.join()

  print(result)
